# 05 — Haystack Integration: Hybrid Pipeline with SimlarDocumentStore

`SimlarDocumentStore` and `SimlarHybridRetriever` are Haystack 2.x components. They slot into any Haystack `Pipeline`, letting you combine simlar with Haystack's embedders, generators, and routers.

**What we cover**
- Loading a HuggingFace dataset and creating Haystack `Document` objects
- Embedding documents with `SentenceTransformersDocumentEmbedder`
- Writing to `SimlarDocumentStore`
- Building a query pipeline: `SentenceTransformersTextEmbedder` → `SimlarHybridRetriever`
- Running queries and inspecting results by news category
- Filtering documents and scoping search with Haystack filter DSL
- Metadata introspection: field types, min/max, unique values
- Updating and deleting documents
- Saving and reloading the store from disk

In [ ]:
%pip install -q datasets haystack-ai sentence-transformers simlar

## Load the dataset

We use [`ag_news`](https://huggingface.co/datasets/fancyzhx/ag_news) — 120,000 news headlines and descriptions spanning four categories: **World**, **Sports**, **Business**, and **Sci/Tech**.

We index the first 500 articles from the training split to keep the demo fast.

In [ ]:
from datasets import load_dataset

LABEL_NAMES = {0: "World", 1: "Sports", 2: "Business", 3: "Sci/Tech"}

ds = load_dataset("fancyzhx/ag_news", split="train[:500]")
texts  = ds["text"]
labels = [LABEL_NAMES[l] for l in ds["label"]]

print(f"Loaded {len(texts)} articles")
print(f"Category distribution: { {k: labels.count(k) for k in LABEL_NAMES.values()} }")
print(f"\nSample: {texts[0][:120]}")

## Create Haystack Documents

Each `Document` carries the article text as `content` and the news category as metadata.
`SimlarDocumentStore` requires that documents arrive **with embeddings already set** — the store itself is embedding-agnostic.

In [ ]:
from haystack import Document

haystack_docs = [
    Document(content=text, meta={"category": label, "pos": i})
    for i, (text, label) in enumerate(zip(texts, labels))
]

print(f"Created {len(haystack_docs)} Haystack documents")
print(f"Example: {haystack_docs[0]}")

## Embed and index documents

`SentenceTransformersDocumentEmbedder` sets the `embedding` field on each document in-place.
We then write the embedded documents into `SimlarDocumentStore`.


In [ ]:
from haystack.components.embedders import SentenceTransformersDocumentEmbedder
from simlar.integrations.haystack.simlar_document_store import SimlarDocumentStore

doc_embedder = SentenceTransformersDocumentEmbedder(
    model="sentence-transformers/all-MiniLM-L6-v2"
)
doc_embedder.warm_up()

embed_result   = doc_embedder.run(documents=haystack_docs)
embedded_docs  = embed_result["documents"]
print(f"Embedded {len(embedded_docs)} documents  |  dim: {len(embedded_docs[0].embedding)}")

store = SimlarDocumentStore(top_k=5)
n_written = store.write_documents(embedded_docs)
print(f"Indexed {n_written} documents  |  store total: {store.count_documents()}")

## Build the query pipeline

The pipeline has two components:
1. `SentenceTransformersTextEmbedder` — encodes the query string into a vector
2. `SimlarHybridRetriever` — receives both the raw query text and the query embedding, runs hybrid search, returns ranked `Document` objects

We connect the embedder's output to the retriever's `query_embedding` input.
The raw query text goes to the retriever as a separate user input when we call `.run()`.

In [ ]:
from haystack import Pipeline
from haystack.components.embedders import SentenceTransformersTextEmbedder
from simlar.integrations.haystack.simlar_retriever import SimlarHybridRetriever

retriever = SimlarHybridRetriever(document_store=store, top_k=5)

query_pipeline = Pipeline()
query_pipeline.add_component(
    "text_embedder",
    SentenceTransformersTextEmbedder(model="sentence-transformers/all-MiniLM-L6-v2"),
)
query_pipeline.add_component("retriever", retriever)
query_pipeline.connect("text_embedder.embedding", "retriever.query_embedding")

print("Query pipeline ready")

## Run queries

Pass the same question to both `text_embedder.text`  and `retriever.query`. The retriever fuses both rankings with RRF and returns the top documents.

In [ ]:
def run_query(question: str) -> None:
    result = query_pipeline.run(
        {
            "text_embedder": {"text": question},
            "retriever":     {"query": question},
        }
    )
    docs = result["retriever"]["documents"]
    print(f"Query : '{question}'")
    print(f"Results ({len(docs)}):")
    for doc in docs:
        cat   = doc.meta.get("category", "?")
        score = doc.meta.get("score", 0.0)
        print(f"  [{cat:8s}]  score={score:.4f}  {doc.content[:90]}")
    print()


run_query("technology startup funding Silicon Valley")
run_query("Olympic Games world record athlete")
run_query("interest rate Federal Reserve inflation")

## Inspect the store

`filter_documents()` with no arguments returns every active document.
Pass a Haystack filter dict to scope results to a subset.

In [ ]:
print(f"Total documents in store : {store.count_documents()}")

# Full breakdown using count_documents_by_filter
for cat in LABEL_NAMES.values():
    n = store.count_documents_by_filter(
        {"operator": "==", "field": "meta.category", "value": cat}
    )
    print(f"  {cat:10s}: {n}")

## Filter documents

`filter_documents(filters)` accepts the standard Haystack filter DSL — the same dict structure used by OpenSearch, Weaviate, and other Haystack document stores.
Supported operators: `==`, `!=`, `>`, `>=`, `<`, `<=`, `in`, `not in`, `AND`, `OR`, `NOT`.

In [ ]:
# All Sports articles
sports_docs = store.filter_documents(
    {"operator": "==", "field": "meta.category", "value": "Sports"}
)
print(f"Sports articles : {len(sports_docs)}")
for doc in sports_docs[:3]:
    print(f"  {doc.content[:100]}")

print()

# Business OR Sci/Tech articles (compound OR filter)
tech_biz_docs = store.filter_documents(
    {
        "operator": "OR",
        "conditions": [
            {"operator": "==", "field": "meta.category", "value": "Business"},
            {"operator": "==", "field": "meta.category", "value": "Sci/Tech"},
        ],
    }
)
print(f"Business + Sci/Tech articles : {len(tech_biz_docs)}")

# Positional range — first 50 articles only
early_docs = store.filter_documents(
    {"operator": "<", "field": "meta.pos", "value": 50}
)
print(f"Articles with pos < 50 : {len(early_docs)}")

## Filtered search

`store.search()` accepts an optional `filters` dict that scopes results to a subset of documents before returning ranked hits.
This is useful when you want hybrid retrieval within a single category or date range.

In [ ]:
import numpy as np
from haystack.components.embedders import SentenceTransformersTextEmbedder

text_embedder = SentenceTransformersTextEmbedder(model="sentence-transformers/all-MiniLM-L6-v2")
text_embedder.warm_up()

question = "stock market earnings profit"
query_embedding = text_embedder.run(text=question)["embedding"]

# Unfiltered — may return results from any category
all_results = store.search(query_text=question, query_embedding=query_embedding, top_k=5)
print("Unfiltered results:")
for doc in all_results:
    print(f"  [{doc.meta['category']:8s}]  score={doc.meta['score']:.4f}  {doc.content[:80]}")

print()

# Business-only
biz_results = store.search(
    query_text=question,
    query_embedding=query_embedding,
    top_k=5,
    filters={"operator": "==", "field": "meta.category", "value": "Business"},
)
print("Business-only results:")
for doc in biz_results:
    print(f"  [{doc.meta['category']:8s}]  score={doc.meta['score']:.4f}  {doc.content[:80]}")

## Metadata introspection

Three helpers let you explore what's in the store without loading every document:

| Method | Returns |
|---|---|
| `get_metadata_fields_info()` | Field names → inferred type (`keyword`, `long`, `float`, …) |
| `get_metadata_field_min_max(field)` | `{"min": …, "max": …}` for a numeric / sortable field |
| `get_metadata_field_unique_values(field)` | All distinct values for a field |
| `count_unique_metadata_by_filter(filters, fields)` | Unique-value count per field, scoped by a filter |

In [ ]:
# Field types inferred from stored documents
print("Metadata schema:")
for field, info in store.get_metadata_fields_info().items():
    print(f"  {field:12s}: {info['type']}")

print()

# Range of the positional field
pos_range = store.get_metadata_field_min_max("meta.pos")
print(f"pos range : {pos_range['min']} – {pos_range['max']}")

print()

# All category labels present in the store
categories = store.get_metadata_field_unique_values("meta.category")
print(f"Unique categories : {sorted(categories)}")

print()

# How many distinct categories appear in Sports vs World articles?
counts = store.count_unique_metadata_by_filter(
    filters={
        "operator": "OR",
        "conditions": [
            {"operator": "==", "field": "meta.category", "value": "Sports"},
            {"operator": "==", "field": "meta.category", "value": "World"},
        ],
    },
    metadata_fields=["meta.category", "meta.pos"],
)
print(f"Unique values in Sports+World slice: {counts}")

## Update metadata

`update_by_filter(filters, meta)` patches the `meta` dict of every matching document in-place.
Useful for bulk tagging, correcting labels, or adding computed fields after indexing.

In [ ]:
# Tag all Sci/Tech articles as reviewed
n_updated = store.update_by_filter(
    filters={"operator": "==", "field": "meta.category", "value": "Sci/Tech"},
    meta={"reviewed": True},
)
print(f"Updated {n_updated} Sci/Tech documents")

# Verify the new field is present
reviewed = store.filter_documents(
    {"operator": "==", "field": "meta.reviewed", "value": True}
)
print(f"Documents with reviewed=True : {len(reviewed)}")
print(f"Example meta: {reviewed[0].meta}")

## Delete documents

The underlying `StreamingHelixIndex` is append-only, so deletions are handled via tombstones — deleted documents are hidden from all queries and `filter_documents()` calls without rebuilding the index.

| Method | Use when |
|---|---|
| `delete_documents(ids)` | You have exact document IDs |
| `delete_by_filter(filters)` | You want to remove a category or range |
| `delete_all_documents()` | Full reset — rebuilds the index from scratch |

In [ ]:
print(f"Before deletion : {store.count_documents()} documents")

# Delete by explicit ID
first_doc_id = store.filter_documents()[0].id
store.delete_documents([first_doc_id])
print(f"After deleting 1 by ID : {store.count_documents()} documents")

# Delete all World articles
n_deleted = store.delete_by_filter(
    {"operator": "==", "field": "meta.category", "value": "World"}
)
print(f"After deleting World articles ({n_deleted} removed) : {store.count_documents()} documents")

# Confirm World is gone from the remaining breakdown
print("\nRemaining category breakdown:")
for cat in LABEL_NAMES.values():
    n = store.count_documents_by_filter(
        {"operator": "==", "field": "meta.category", "value": cat}
    )
    print(f"  {cat:10s}: {n}")

## Persistence: save and reload

`save(path)` writes index and all document metadata to a directory.
`SimlarDocumentStore.load(path)` reconstructs an identical store 

This lets you build and embed once, then serve the store across multiple sessions without re-running the embedder.

In [ ]:
import tempfile
from pathlib import Path

with tempfile.TemporaryDirectory() as tmp:
    save_path = Path(tmp) / "ag_news_store"

    # Persist
    store.save(save_path)
    saved_files = list(save_path.iterdir())
    print(f"Saved to {save_path}")
    print(f"Files written: {[f.name for f in saved_files]}")

    # Reload into a fresh object
    reloaded_store = SimlarDocumentStore.load(save_path)
    print(f"\nReloaded store: {reloaded_store.count_documents()} documents")

    # Sanity-check: category counts must match
    for cat in LABEL_NAMES.values():
        orig = store.count_documents_by_filter(
            {"operator": "==", "field": "meta.category", "value": cat}
        )
        reloaded = reloaded_store.count_documents_by_filter(
            {"operator": "==", "field": "meta.category", "value": cat}
        )
        status = "OK" if orig == reloaded else "MISMATCH"
        print(f"  {cat:10s}: orig={orig}  reloaded={reloaded}  [{status}]")

    # Search works on the reloaded store
    hits = reloaded_store.search(
        query_text="Olympic Games world record",
        query_embedding=text_embedder.run(text="Olympic Games world record")["embedding"],
        top_k=3,
    )
    print(f"\nSearch on reloaded store ({len(hits)} hits):")
    for doc in hits:
        print(f"  [{doc.meta['category']:8s}]  {doc.content[:90]}")